# 02 · Preprocessing (Phase 3)

Rasters (P3.1–P3.3) and OSM vectors (P3.4–P3.6) brought onto one grid in the project CRS:

1. run the preprocessing (skips outputs that already exist);
2. **table of every processed layer:** CRS, resolution, shape, nodata % (P3.7);
3. **before / after** plots for rasters and vectors (P3.7);
4. **overlay check:** roads, water and buildings over LULC, no shifts (P3.9).

Correctness is checked independently in `scripts/verify_preprocessed.py` and `scripts/verify_vectors.py`.

In [ ]:
import json

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
from matplotlib.colors import ListedColormap
from rasterio.features import geometry_mask
from rasterio.warp import transform as warp_points

from src.config import load_config
from src.download.dem import dem_path
from src.download.lulc import esri_path
from src.download.osm import layers_for, osm_path, snapshots
from src.io_utils import setup_logging
from src.preprocess.raster import load_reference_grid, preprocess_rasters, processed_dir
from src.preprocess.vector import preprocess_vectors, processed_osm_path

cfg = load_config()
setup_logging(cfg, "preprocess")
print(cfg["aoi"]["name"], cfg.crs.to_string())

## 1. Run the preprocessing

In [ ]:
preprocess_rasters(cfg)  # skips existing outputs
if not all(processed_osm_path(cfg, s, l).exists() for s in snapshots(cfg) for l in layers_for(s)):
    preprocess_vectors(cfg)
ref = load_reference_grid(cfg)
print(json.dumps(ref.to_json(), indent=1))

## 2. Every processed layer (P3.7)

In [ ]:
aoi = cfg.aoi_projected.geometry
rows = []
for f in sorted(processed_dir(cfg).glob("*.tif")):
    with rasterio.open(f) as ds:
        a = ds.read(1)
        inside = ~geometry_mask(aoi, a.shape, ds.transform)
        nod = a == ds.nodata
        rows.append({"layer": f.name, "type": "raster", "crs": ds.crs.to_string(), "resolution": f"{ds.res[0]:g} m",
                     "shape / features": f"{ds.height} x {ds.width}", "nodata % (grid)": round(100 * nod.mean(), 3),
                     "nodata % (AOI)": round(100 * nod[inside].mean(), 3), "same grid as reference": ds.transform == ref.transform and ds.shape == ref.shape})
for s in snapshots(cfg):
    for l in layers_for(s):
        g = gpd.read_file(processed_osm_path(cfg, s, l))
        rows.append({"layer": f"osm/{s}/{l}.gpkg", "type": "vector", "crs": g.crs.to_string(), "resolution": "-",
                     "shape / features": len(g), "nodata % (grid)": "-", "nodata % (AOI)": "-",
                     "same grid as reference": bool(g.within(gpd.GeoSeries([__import__('shapely').box(*ref.bounds)], crs=g.crs).iloc[0].buffer(0.01)).all())})
table = pd.DataFrame(rows)
table

## 3a. Before / after: rasters

Raw downloads (left, each in its own CRS and grid) vs the aligned 10 m outputs (right).

In [ ]:
ESRI = {1: "#419bdf", 2: "#397d49", 4: "#7a87c6", 5: "#e49635", 7: "#c4281b", 8: "#a59b8f", 11: "#e3e2c3"}
lut = np.zeros(256, np.uint8)
for i, c in enumerate(sorted(ESRI)):
    lut[c] = i + 1
cmap = ListedColormap(["white"] + [ESRI[c] for c in sorted(ESRI)])
year = cfg["years"]["baseline"]
fig, axes = plt.subplots(2, 2, figsize=(13, 12))
with rasterio.open(esri_path(cfg, year)) as ds:
    b = ds.bounds
    axes[0, 0].imshow(lut[ds.read(1)], cmap=cmap, vmin=0, vmax=len(ESRI), extent=(b.left, b.right, b.bottom, b.top), interpolation="nearest")
    axes[0, 0].set_title(f"raw ESRI {year}: {ds.crs.to_string()}, origin ({b.left:.0f}, {b.top:.0f})")
with rasterio.open(processed_dir(cfg) / f"lulc_esri_{year}.tif") as ds:
    b = ds.bounds
    axes[0, 1].imshow(lut[ds.read(1)], cmap=cmap, vmin=0, vmax=len(ESRI), extent=(b.left, b.right, b.bottom, b.top), interpolation="nearest")
    axes[0, 1].set_title(f"processed: reference grid, origin ({b.left:.0f}, {b.top:.0f})")
with rasterio.open(dem_path(cfg)) as ds:
    b = ds.bounds
    im = axes[1, 0].imshow(np.ma.masked_equal(ds.read(1), ds.nodata), cmap="terrain", extent=(b.left, b.right, b.bottom, b.top))
    axes[1, 0].set_title(f"raw DEM: {ds.crs.to_string()}, {ds.res[0] * 3600:.0f} arc-second")
    fig.colorbar(im, ax=axes[1, 0], shrink=0.7, label="m")
with rasterio.open(processed_dir(cfg) / "slope.tif") as ds:
    b = ds.bounds
    im = axes[1, 1].imshow(ds.read(1), cmap="magma", vmin=0, vmax=20, extent=(b.left, b.right, b.bottom, b.top))
    axes[1, 1].set_title("processed slope (Horn, degrees)")
    fig.colorbar(im, ax=axes[1, 1], shrink=0.7, label="deg")
for ax in axes.ravel():
    ax.set_axis_off()
plt.tight_layout()

## 3b. Before / after: OSM vectors

Feature counts and road classes after cleaning (clip to the grid, `make_valid`, drop non-drivable roads and tiny / outlier buildings).

In [ ]:
rows = []
for s in snapshots(cfg):
    for l in layers_for(s):
        raw = gpd.read_file(osm_path(cfg, s, l))
        out = gpd.read_file(processed_osm_path(cfg, s, l))
        extra = ""
        if l == "roads":
            extra = ", ".join(f"{k} {v / 1000:,.0f} km" for k, v in out.length.groupby(out["road_class"]).sum().items())
        if l == "buildings":
            extra = f"median {out['area_m2'].median():.0f} m²"
        rows.append({"snapshot": s, "layer": l, "raw features": len(raw), "raw CRS": raw.crs.to_string(),
                     "clean features": len(out), "clean CRS": out.crs.to_string(), "notes": extra})
pd.DataFrame(rows)

## 4. Overlay check (P3.9)

If a layer were shifted, roads would run beside the built-up strips, lakes would sit off the blue ESRI water and buildings would fall on non-built pixels.

In [ ]:
with rasterio.open(processed_dir(cfg) / f"lulc_esri_{year}.tif") as ds:
    lulc = ds.read(1)
snap = str(cfg["osm"]["snapshot_baseline"])[:4]
roads = gpd.read_file(processed_osm_path(cfg, snap, "roads"))
water = gpd.read_file(processed_osm_path(cfg, snap, "water"))
bldg = gpd.read_file(processed_osm_path(cfg, snap, "buildings"))
b = ref.bounds
fig, axes = plt.subplots(1, 2, figsize=(16, 8))
axes[0].imshow(lut[lulc], cmap=cmap, vmin=0, vmax=len(ESRI), extent=(b[0], b[2], b[1], b[3]), interpolation="nearest")
roads[roads.road_class == "major"].plot(ax=axes[0], color="black", linewidth=0.6)
water[water.kind == "water_body"].boundary.plot(ax=axes[0], color="#08306b", linewidth=0.6)
cfg.aoi_projected.boundary.plot(ax=axes[0], color="black", linewidth=1)
axes[0].set_title(f"{snap}: major roads (black), OSM lakes (navy) over ESRI {year}")
x, y = warp_points("EPSG:4326", cfg.crs, [77.6177], [12.9082])  # Madiwala Lake
z = (x[0] - 700, x[0] + 700, y[0] - 700, y[0] + 700)
r0, c0 = int((b[3] - z[3]) / 10), int((z[0] - b[0]) / 10)
axes[1].imshow(lut[lulc[r0:r0 + 140, c0:c0 + 140]], cmap=cmap, vmin=0, vmax=len(ESRI), extent=z, interpolation="nearest")
bldg.cx[z[0]:z[1], z[2]:z[3]].boundary.plot(ax=axes[1], color="black", linewidth=0.3)
roads.cx[z[0]:z[1], z[2]:z[3]].plot(ax=axes[1], color="#252525", linewidth=0.8)
water.cx[z[0]:z[1], z[2]:z[3]].boundary.plot(ax=axes[1], color="#08306b", linewidth=1.2)
axes[1].set_xlim(z[0], z[1]); axes[1].set_ylim(z[2], z[3])
axes[1].set_title("Madiwala Lake, 1.4 km: buildings, roads and lake outline")
for ax in axes:
    ax.set_axis_off()
plt.tight_layout()

**Reading the overlay:** the lake outline sits on the ESRI water, roads run through built-up land, and buildings fall on built pixels. OSM lakes are offset about 5–10 m from Sentinel-2 products (`scripts/verify_vectors.py`; the same in the raw files), which is under one pixel.